# ⚛️ Primer Circuito Cuántico — Ciudad Robot

Este notebook introduce los conceptos básicos de computación cuántica aplicados
al módulo **quantum-core** de Ciudad Robot.

## ¿Qué aprenderás?
1. Qué es un qubit y cómo representarlo con NumPy
2. Cómo construir el circuito cuántico de semáforo
3. Cómo simular el circuito con `LocalSimulator`
4. Cómo interpretar los resultados de medición

## 1. Configuración del entorno

Asegúrate de tener instaladas las dependencias:
```bash
pip install numpy pytest
```

In [ ]:
import sys
import os

# Añadir la raíz del repositorio al path para importar quantum-core
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import numpy as np
print('NumPy version:', np.__version__)
print('Entorno listo ✅')

## 2. ¿Qué es un qubit?

Un qubit es la unidad básica de información cuántica. A diferencia de un bit clásico
(0 o 1), un qubit puede estar en **superposición** de ambos estados.

Representamos un qubit como un vector complejo de dos componentes:
$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$$

donde $|\alpha|^2 + |\beta|^2 = 1$.

In [ ]:
# Estado |0> (qubit en estado base)
qubit_0 = np.array([1.0, 0.0], dtype=complex)
print('Estado |0>:', qubit_0)

# Estado |1>
qubit_1 = np.array([0.0, 1.0], dtype=complex)
print('Estado |1>:', qubit_1)

# Superposición |+> = (|0> + |1>) / sqrt(2)  — puerta Hadamard aplicada a |0>
qubit_plus = np.array([1.0, 1.0], dtype=complex) / np.sqrt(2)
print('Estado |+>:', qubit_plus)
print('Probabilidades:', np.abs(qubit_plus)**2)

## 3. Circuito cuántico de semáforo

El módulo `circuits.traffic_light_circuit` construye un circuito cuántico
que representa el estado de un semáforo. Usamos **matrices numpy** para
simular las puertas cuánticas.

In [ ]:
from quantum_core.circuits.traffic_light_circuit import build_traffic_light_circuit

# Construir el circuito con parámetros por defecto
circuito = build_traffic_light_circuit()
print('Circuito construido:')
print(' Nombre  :', circuito.get('nombre', 'semaforo_cuantico'))
print(' Qubits  :', circuito.get('n_qubits', 3))
print(' Puertas :', circuito.get('n_puertas', 0))

## 4. Simular el circuito

El `LocalSimulator` ejecuta el circuito en memoria sin necesidad de hardware cuántico real.

In [ ]:
from quantum_core.backends.local_simulator import LocalSimulator

simulador = LocalSimulator(shots=1024)
resultado = simulador.run(circuito)

print('Backend:', resultado.get('backend'))
print('Shots  :', resultado.get('shots'))
print('Conteos:')
conteos = resultado.get('conteos', {})
for bits, count in sorted(conteos.items(), key=lambda x: -x[1]):
    barra = '█' * (count // 32)
    print(f'  |{bits}> : {count:4d}  {barra}')

## 5. Interpretar resultados

En nuestro circuito de semáforo, el **último bit** del estado medido representa
el estado del semáforo:
- `1` → **Verde** (paso permitido)
- `0` → **Rojo** (stop)

Si la probabilidad de `1` es:
- `> 55%` → Semáforo en **verde**
- `< 45%` → Semáforo en **rojo**
- Entre 45% y 55% → **Amarillo** (indeterminado)

In [ ]:
total = sum(conteos.values())
votos_verde = sum(cnt for bits, cnt in conteos.items() if bits and bits[-1] == '1')
prob_verde = votos_verde / total if total > 0 else 0

if prob_verde > 0.55:
    estado = '🟢 VERDE — Paso permitido'
elif prob_verde < 0.45:
    estado = '🔴 ROJO — Stop'
else:
    estado = '🟡 AMARILLO — Precaución'

print(f'Probabilidad verde : {prob_verde:.1%}')
print(f'Estado del semáforo: {estado}')

## 🏁 Resumen

En este notebook has aprendido a:
1. ✅ Representar qubits con NumPy
2. ✅ Construir un circuito cuántico de semáforo
3. ✅ Simular el circuito con `LocalSimulator`
4. ✅ Interpretar los resultados de medición

**Siguiente paso:** → `02_qaoa_optimizacion_trafico.ipynb`

Allí exploraremos el algoritmo QAOA para optimizar el flujo de tráfico en la ciudad.